In [ ]:
!pip install kaggle-environments stable-baselines3

In [ ]:
%%writefile heuristic.py
"""Kaggriculture heuristic agent -- TOP PLAYER STRATEGY (v3).

Strategy reverse-engineered from top leaderboard replays (103k-126k scores):
  Day 0:   Buy 5-6 hands. Build PASTURE(s). Buy SHEEP+COW. Plant WHEAT for feed.
  Day 1-7: Water crops, feed/care animals, collect fertilizer. Buy more land+pastures.
  Day 7+:  Expand: more PASTURE, more SHEEP+COW, more WHEAT for feed loop.
  Mid:     14 PASTUREs with SHEEP+COW, CARE daily, COLLECT_FERTILIZER daily.
  Late:    Buy WHEAT from market to sell back for arbitrage, HIRE max hands.
"""
import math

# ── Market price model ────────────────────────────────────────────────────────
_FUNCS = {
    "linear": lambda x: float(x),
    "sq":     lambda x: float(x) * float(x),
    "sqrt":   lambda x: math.sqrt(x),
    "log":    lambda x: math.log(1.0 + x),
    "log10":  lambda x: math.log10(1.0 + x),
}
MARKET_PARAMS = {
    "WHEAT":      (25,  10000, 400, "sqrt",   0.80, "log",    0.20),
    "CARROT":     (35,  10000, 450, "log",    0.20, "sqrt",   0.70),
    "TOMATO":     (60,  10000, 200, "linear", 0.40, "sqrt",   0.60),
    "STRAWBERRY": (120, 10000, 100, "sqrt",   0.70, "linear", 1.60),
    "MELON":      (250, 10000, 300, "log",    0.20, "sq",     3.60),
    "EGG":        (50,  10000, 332, "linear", 0.40, "log",    0.20),
    "MILK":       (160, 10000, 122, "sqrt",   0.60, "linear", 1.60),
    "WOOL":       (200, 10000, 105, "log",    0.20, "sq",     3.20),
    "FERTILIZER": (100, 10000, 200, "linear", 0.40, "linear", 0.40),
}
PRODUCTS = list(MARKET_PARAMS.keys())
SELLABLE = PRODUCTS
SELL_PRODUCE = PRODUCTS


def price(resource, inv):
    base, I0, T, bf, bt, af, at = MARKET_PARAMS[resource]
    if inv < I0:
        f = _FUNCS[bf]
        p = base + (bt * base / f(T)) * f(I0 - inv)
    elif inv > I0:
        f = _FUNCS[af]
        p = base - (at * base / f(T)) * f(inv - I0)
    else:
        p = float(base)
    return max(1, int(round(p)))


def sell_revenue(resource, inv, qty):
    rev = 0
    for _ in range(int(qty)):
        p = price(resource, inv)
        rev += p
        if p > 1:
            inv += 1
    return rev, inv


def marginal_revenue(resource, inv, qty):
    if qty <= 1:
        return price(resource, inv)
    _, inv2 = sell_revenue(resource, inv, qty - 1)
    return price(resource, inv2)


def buy_cost(resource, inv, qty):
    cost = 0
    for _ in range(int(qty)):
        inv -= 1
        cost += price(resource, inv)
    return cost, inv


def units_sellable_above(resource, inv, reserve):
    if reserve <= 1:
        return 10 ** 6
    n = 0
    while n < 2000:
        p = price(resource, inv)
        if p < reserve:
            break
        n += 1
        if p > 1:
            inv += 1
    return n


_RESERVE_FRAC = {
    "WHEAT": 0.72, "CARROT": 0.70, "TOMATO": 0.70, "EGG": 0.72,
    "MILK": 0.72, "WOOL": 0.72, "STRAWBERRY": 0.72, "MELON": 0.72,
    "FERTILIZER": 0.72,
}

# ── Specs ─────────────────────────────────────────────────────────────────────
CROP_SPECS = {
    "WHEAT":      dict(seed=10,  first=1,  maxday=2,  peak=2, occ=1),
    "CARROT":     dict(seed=20,  first=2,  maxday=3,  peak=4, occ=2),
    "TOMATO":     dict(seed=50,  first=8,  maxday=8,  peak=4, occ=8),
    "STRAWBERRY": dict(seed=100, first=10, maxday=10, peak=4, occ=10),
    "MELON":      dict(seed=80,  first=10, maxday=12, peak=6, occ=10),
}
ANIMAL_SPECS = {
    "GOOSE": dict(cost=300,  build="BUILD_COOP",    structure="COOP",    product="EGG",  interval=1, first=4),
    "COW":   dict(cost=1000, build="BUILD_PASTURE", structure="PASTURE", product="MILK", interval=2, first=8),
    "SHEEP": dict(cost=1200, build="BUILD_PASTURE", structure="PASTURE", product="WOOL", interval=3, first=6),
}
ANIMAL_ITEMS = ("GOOSE", "COW", "SHEEP")
CROPS = list(CROP_SPECS.keys())


# ── Market helpers ─────────────────────────────────────────────────────────────
def plan_sells(shed, market_inv, day, hour, total_days, hold=None):
    """Decide what to sell this turn."""
    orders = []
    hold = hold or {}
    for product in SELL_PRODUCE:
        qty = int(shed.get(product, 0))
        reserve = hold.get(product, 0)
        qty = max(0, qty - reserve)
        if qty <= 0:
            continue
        inv = int(market_inv.get(product, MARKET_PARAMS[product][1]))
        reserve_price = _RESERVE_FRAC.get(product, 0.70) * MARKET_PARAMS[product][0]
        sell_n = min(qty, units_sellable_above(product, inv, reserve_price))
        # Late game: sell everything
        if day >= total_days - 3:
            sell_n = qty
        if sell_n > 0:
            orders.append(["SELL", product, sell_n])
    return orders


def feed_hold(animal_count, shed_wheat, days_buffer=2):
    return min(shed_wheat, animal_count * days_buffer)


# ── Farm scan ─────────────────────────────────────────────────────────────────
def _scan(me, day):
    out = dict(water=[], feed=[], harvest_crop=[], harvest_animal=[],
               collect=[], weeds=[], structures_empty=[], care=[],
               collect_fertilizer=[])
    tiles = me.get("tiles") or []
    for y, row in enumerate(tiles):
        for x, t in enumerate(row):
            if not isinstance(t, dict):
                continue
            kind = t.get("kind")
            if kind == "PLANT":
                if not t.get("watered_today"):
                    out["water"].append((x, y))
                if t.get("yield_units", 0) > 0:
                    out["harvest_crop"].append((x, y))
            elif kind in ("COOP", "PASTURE"):
                if t.get("animal"):
                    if not t.get("fed_today"):
                        out["feed"].append((x, y))
                    if t.get("fertilizer_available"):
                        out["collect_fertilizer"].append((x, y))
                    out["care"].append((x, y))
                    if t.get("yield_units", 0) > 0:
                        out["harvest_animal"].append((x, y))
                else:
                    out["structures_empty"].append(((x, y), kind))
            elif kind == "WEED":
                out["weeds"].append((x, y))
    return out


def _free_cells(me, board):
    tiles = me.get("tiles") or []
    free = []
    for y, row in enumerate(tiles):
        for x, t in enumerate(row):
            if t is None:
                free.append((x, y))
    return free


def _count_animals(me, private=None):
    n = 0
    tiles = me.get("tiles") or []
    for row in tiles:
        for t in row:
            if isinstance(t, dict) and t.get("kind") in ("COOP", "PASTURE") and t.get("animal"):
                n += 1
    if private:
        shed = private.get("shed") or {}
        n += int(shed.get("COW", 0))
        n += int(shed.get("GOOSE", 0))
        n += int(shed.get("SHEEP", 0))
        for inv in (private.get("inventories") or []):
            n += int(inv.get("COW", 0))
            n += int(inv.get("GOOSE", 0))
            n += int(inv.get("SHEEP", 0))
    return n


def _count_structures(me, kind):
    n = 0
    tiles = me.get("tiles") or []
    for row in tiles:
        for t in row:
            if isinstance(t, dict) and t.get("kind") == kind:
                n += 1
    return n


def _count_planted(me):
    n = 0
    tiles = me.get("tiles") or []
    for row in tiles:
        for t in row:
            if isinstance(t, dict) and t.get("kind") == "PLANT":
                n += 1
    return n


def _land_cost(me):
    n = len(me.get("unlocked_quadrants", ["NW"])) - 1
    costs = [1000, 2000, 3000]
    return costs[n] if n < len(costs) else 999999


# ── Movement helpers ──────────────────────────────────────────────────────────
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def step_toward(pos, target):
    x, y = pos
    tx, ty = target
    dx, dy = tx - x, ty - y
    if dx == 0 and dy == 0:
        return None
    if abs(dx) >= abs(dy):
        return "EAST" if dx > 0 else "WEST"
    return "SOUTH" if dy > 0 else "NORTH"


def _nearest(pos, cells):
    if not cells:
        return None
    return min(cells, key=lambda c: manhattan(pos, c))


def shed_adjacent_cells(board, me):
    """Cells adjacent to the shed (top-left corner of the farm, near 0,0)."""
    tiles = me.get("tiles") or []
    shed_pos = (0, 0)
    candidates = []
    for dy in range(min(3, board)):
        for dx in range(min(3, board)):
            t = tiles[dy][dx] if dy < len(tiles) and dx < len(tiles[dy]) else "LOCKED"
            if t != "LOCKED":
                candidates.append((dx, dy))
    return candidates or [(0, 0)]


def hire_cost(n_already_hired):
    fib = [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]
    return fib[min(n_already_hired, len(fib) - 1)]


# ── TOP STRATEGY: Day-0 burst plan ───────────────────────────────────────────
def _day0_plan(money, free_tiles, quadrants):
    """
    Top players spend everything on Day 0:
    - Build 6 PASTUREs immediately
    - Buy 2 SHEEP + 2 COW
    - Plant WHEAT on remaining tiles for feed
    """
    orders = []
    money_left = money

    # Buy WHEAT seeds for feed crops
    wheat_tiles = min(free_tiles, 7)
    wheat_cost = wheat_tiles * 10
    if money_left > wheat_cost + 100:
        orders.append(["BUY_SEED", "WHEAT", wheat_tiles])
        money_left -= wheat_cost

    # Buy SHEEP (WOOL=200/unit, high value)
    n_sheep = min(2, int((money_left - 500) // 1200))
    if n_sheep > 0:
        orders.append(["BUY_ANIMAL", "SHEEP", n_sheep])
        money_left -= n_sheep * 1200

    # Buy COW (MILK=160/2 turns)
    n_cow = min(2, int((money_left - 300) // 1000))
    if n_cow > 0:
        orders.append(["BUY_ANIMAL", "COW", n_cow])
        money_left -= n_cow * 1000

    # Hire hands
    for i in range(5):
        c = hire_cost(i)
        if money_left > c + 200:
            orders.append(["HIRE"])
            money_left -= c

    return orders


# ── Task assignment ──────────────────────────────────────────────────────────
TASK_PRIORITY = {
    "feed": 10,
    "care": 9,
    "collect_fertilizer": 8,
    "harvest_animal": 7,
    "harvest_crop": 6,
    "water": 5,
    "build_pasture": 4,
    "place_animal": 3,
    "plant": 2,
    "weed": 1,
}


def _unit_op(pos, task, private, unit_idx, board, me, shed_wheat):
    """Convert a task assignment to an actual operation."""
    kind = task[0]
    target = task[1] if len(task) > 1 else None
    pos = tuple(pos)

    if kind == "feed":
        if pos == tuple(target):
            return ["FEED"]
        st = step_toward(pos, target)
        return [st] if st else ["FEED"]

    elif kind == "care":
        if pos == tuple(target):
            return ["CARE"]
        st = step_toward(pos, target)
        return [st] if st else ["CARE"]

    elif kind == "collect_fertilizer":
        if pos == tuple(target):
            return ["COLLECT_FERTILIZER"]
        st = step_toward(pos, target)
        return [st] if st else ["COLLECT_FERTILIZER"]

    elif kind == "harvest_animal":
        if pos == tuple(target):
            return ["HARVEST"]
        st = step_toward(pos, target)
        return [st] if st else ["HARVEST"]

    elif kind == "harvest_crop":
        if pos == tuple(target):
            return ["HARVEST"]
        st = step_toward(pos, target)
        return [st] if st else ["HARVEST"]

    elif kind == "water":
        if pos == tuple(target):
            return ["WATER"]
        st = step_toward(pos, target)
        return [st] if st else ["WATER"]

    elif kind == "build_pasture":
        if pos == tuple(target):
            return ["BUILD_PASTURE"]
        st = step_toward(pos, target)
        return [st] if st else ["BUILD_PASTURE"]

    elif kind == "place_animal":
        animal, target_pos = target
        if pos == tuple(target_pos):
            return ["PLACE", animal]
        st = step_toward(pos, target_pos)
        return [st] if st else ["PLACE", animal]

    elif kind == "plant":
        crop, target_pos = target
        if pos == tuple(target_pos):
            return ["PLANT", crop]
        st = step_toward(pos, target_pos)
        return [st] if st else ["PLANT", crop]

    elif kind == "weed":
        if pos == tuple(target):
            return ["DIG"]
        st = step_toward(pos, target)
        return [st] if st else ["DIG"]

    return ["PASS"]


def _build_tasks(scan, me, private, free_cells, day, total_days):
    """Build prioritized task list based on current farm state."""
    tasks = []

    # Priority 1: Feed animals (critical — miss 2 days = animal escapes)
    for cell in scan["feed"]:
        tasks.append(("feed", cell))

    # Priority 2: CARE animals (generates fertilizer)
    for cell in scan["care"]:
        tasks.append(("care", cell))

    # Priority 3: Collect fertilizer
    for cell in scan["collect_fertilizer"]:
        tasks.append(("collect_fertilizer", cell))

    # Priority 4: Harvest animals
    for cell in scan["harvest_animal"]:
        tasks.append(("harvest_animal", cell))

    # Priority 5: Harvest crops
    for cell in scan["harvest_crop"]:
        tasks.append(("harvest_crop", cell))

    # Priority 6: Water crops
    for cell in scan["water"]:
        tasks.append(("water", cell))

    # Priority 7: Build PASTUREs on empty tiles
    if day < total_days - 10:
        for cell in free_cells[:4]:
            tasks.append(("build_pasture", cell))

    # Priority 8: Place animals from shed into structures
    shed = (private.get("shed") or {})
    invs = (private.get("inventories") or [])
    # Collect all unplaced animals
    unplaced = {}
    for a in ANIMAL_ITEMS:
        n = int(shed.get(a, 0))
        for inv in invs:
            n += int(inv.get(a, 0))
        if n > 0:
            unplaced[a] = n

    if unplaced:
        for (cell, kind) in scan["structures_empty"]:
            for a, n in list(unplaced.items()):
                if n <= 0:
                    continue
                if kind == "COOP" and a == "GOOSE":
                    tasks.append(("place_animal", (a, cell)))
                    unplaced[a] -= 1
                    break
                elif kind == "PASTURE" and a in ("COW", "SHEEP"):
                    tasks.append(("place_animal", (a, cell)))
                    unplaced[a] -= 1
                    break

    # Priority 9: Plant WHEAT on free cells for feed
    seeds = private.get("seeds") or {}
    wheat_seeds = int(seeds.get("WHEAT", 0))
    planted_count = _count_planted(me)
    if wheat_seeds > 0 and day < total_days - 5:
        for cell in free_cells[:wheat_seeds]:
            tasks.append(("plant", ("WHEAT", cell)))

    # Priority 10: Remove weeds
    for cell in scan["weeds"]:
        tasks.append(("weed", cell))

    return tasks


def _assign_tasks(positions, tasks, invs):
    """Greedy nearest-task assignment."""
    assignment = {}
    available_tasks = list(tasks)
    assigned_tasks = set()

    for i, pos in enumerate(positions):
        best_task = None
        best_dist = 999999
        best_idx = -1
        for j, task in enumerate(available_tasks):
            if j in assigned_tasks:
                continue
            kind = task[0]
            target = task[1] if len(task) > 1 else None
            if target is None:
                continue
            if isinstance(target, tuple) and len(target) == 2 and isinstance(target[0], int):
                dist = manhattan(pos, target)
            elif isinstance(target, tuple) and len(target) == 2:
                # (animal, cell) format
                dist = manhattan(pos, target[1])
            else:
                dist = 0
            # Adjust by priority
            priority = TASK_PRIORITY.get(kind, 0)
            score = dist - priority * 3
            if score < best_dist:
                best_dist = score
                best_task = task
                best_idx = j
        if best_task is not None:
            assignment[i] = best_task
            assigned_tasks.add(best_idx)
    return assignment


# ── Market order builder ──────────────────────────────────────────────────────
def _make_market_orders(obs, player=0):
    """Build market orders mimicking top-player strategy."""
    me = obs["farms"][player]
    private = obs.get("private") or {}
    market = obs.get("market") or {}
    minv = market.get("inventory") or {}
    shed = private.get("shed") or {}
    seeds = private.get("seeds") or {}
    day = int(obs.get("day", 0))
    hour = int(obs.get("hour", 0))
    total_days = 30
    money = me.get("money", 0)
    quadrants = me.get("unlocked_quadrants") or ["NW"]
    free_cells = _free_cells(me, len(me.get("tiles") or []) or 10)
    animals = _count_animals(me, private)
    hires_today = int(me.get("hires_today", 0))

    orders = []

    # 1. Sell produce
    hold = {"WHEAT": feed_hold(animals, int(shed.get("WHEAT", 0)))}
    orders += plan_sells(shed, minv, day, hour, total_days, hold)
    money_left = money

    # 2. Emergency wheat buy for animal feed
    if animals > 0 and int(shed.get("WHEAT", 0)) < animals * 2 and day < total_days - 1:
        need = animals * 2 - int(shed.get("WHEAT", 0))
        c, _ = buy_cost("WHEAT", minv.get("WHEAT", 10000), need)
        if money_left > c + 100:
            orders.append(["BUY_PRODUCT", "WHEAT", need])
            money_left -= c

    # 3. Hire hands aggressively (top players hire 8-14/day)
    if hour <= 1:
        # Day 0: hire 5, later scale by animal+work count
        if day == 0:
            target_hands = 5
        else:
            n_pastures = _count_structures(me, "PASTURE") + _count_structures(me, "COOP")
            target_hands = min(14, max(5, n_pastures + 3))
        while hires_today < target_hands:
            c = hire_cost(hires_today)
            if money_left > c + 300:
                orders.append(["HIRE"])
                money_left -= c
                hires_today += 1
            else:
                break

    # 4. Day 0: buy SHEEP+COW+WHEAT seeds
    if day == 0 and hour <= 2:
        # Buy wheat seeds for feed farming
        wheat_tiles = min(len(free_cells), 7)
        if wheat_tiles > 0 and money_left > wheat_tiles * 10 + 200:
            orders.append(["BUY_SEED", "WHEAT", wheat_tiles])
            money_left -= wheat_tiles * 10

        # Buy SHEEP first (WOOL is highest value)
        if money_left > 1200 + 500:
            n = min(2, int((money_left - 500) // 1200))
            if n > 0:
                orders.append(["BUY_ANIMAL", "SHEEP", n])
                money_left -= n * 1200

        # Buy COW
        if money_left > 1000 + 300:
            n = min(2, int((money_left - 300) // 1000))
            if n > 0:
                orders.append(["BUY_ANIMAL", "COW", n])
                money_left -= n * 1000

    # 5. Ongoing: fill empty structures with animals
    scan = _scan(me, day)
    if day > 0 and hour <= 2 and scan["structures_empty"] and day < total_days - 5:
        for (cell, kind) in scan["structures_empty"][:4]:
            if kind == "PASTURE":
                # Pick SHEEP or COW based on market
                milk_p = price("MILK", minv.get("MILK", 10000)) * 0.5
                wool_p = price("WOOL", minv.get("WOOL", 10000)) / 3.0
                a = "SHEEP" if wool_p > milk_p else "COW"
                cost = ANIMAL_SPECS[a]["cost"]
                if money_left > cost + 300:
                    orders.append(["BUY_ANIMAL", a, 1])
                    money_left -= cost
            elif kind == "COOP":
                cost = ANIMAL_SPECS["GOOSE"]["cost"]
                if money_left > cost + 300:
                    orders.append(["BUY_ANIMAL", "GOOSE", 1])
                    money_left -= cost

    # 6. Buy wheat seeds if we have free tiles and no seeds
    if day < total_days - 5 and free_cells and int(seeds.get("WHEAT", 0)) < 3:
        n = min(7, len(free_cells))
        cost = n * 10
        if money_left > cost + 300:
            orders.append(["BUY_SEED", "WHEAT", n])
            money_left -= cost

    # 7. Buy more land when farm is full
    if len(quadrants) < 4 and day < total_days - 8:
        land_cost = _land_cost(me)
        if len(free_cells) < 3 and money_left > land_cost + 500:
            orders.append(["BUY_LAND"])
            money_left -= land_cost

    return orders[:10]


# ── Main agent ────────────────────────────────────────────────────────────────
def _agent(obs):
    player = obs["player"]
    day = int(obs.get("day", 0))
    hour = int(obs.get("hour", 0))
    me = obs["farms"][player]
    private = obs.get("private") or {}
    market = obs.get("market") or {}
    minv = market.get("inventory") or {}
    tiles = me.get("tiles") or []
    board = len(tiles) or 10
    total_days = 30
    money = me.get("money", 0)
    shed = private.get("shed") or {}
    seeds = private.get("seeds") or {}
    invs = private.get("inventories") or [{}]

    scan = _scan(me, day)
    free_cells = _free_cells(me, board)
    animals = _count_animals(me, private)

    orders = _make_market_orders(obs, player)

    # Build task list
    tasks = _build_tasks(scan, me, private, free_cells, day, total_days)

    # Assign tasks to farmer + hands
    positions = [tuple(me.get("farmer") or (4, 4))]
    positions += [tuple(h) for h in (me.get("hands") or [])]
    ops_out = [None] * len(positions)

    # Drop if carrying too much
    sheds = shed_adjacent_cells(board, me)
    for i, pos in enumerate(positions):
        inv = invs[i] if i < len(invs) else {}
        total_carry = sum(v for k, v in inv.items() if k not in ANIMAL_ITEMS)
        if total_carry >= 5:
            pos_t = tuple(pos)
            if pos_t in [tuple(c) for c in sheds]:
                ops_out[i] = ["DROP"]
            else:
                tgt = min(sheds, key=lambda c: manhattan(pos, c))
                st = step_toward(pos, tgt)
                ops_out[i] = [st] if st else ["DROP"]

    # Assign remaining workers to tasks
    free_units = [i for i in range(len(positions)) if ops_out[i] is None]
    free_positions = [positions[i] for i in free_units]
    free_invs = [invs[i] if i < len(invs) else {} for i in free_units]
    assignment = _assign_tasks(free_positions, tasks, free_invs)

    shed_wheat = int(shed.get("WHEAT", 0))
    for local_i, task in assignment.items():
        gi = free_units[local_i]
        ops_out[gi] = _unit_op(positions[gi], task, private, gi, board, me, shed_wheat)

    for i in range(len(ops_out)):
        if ops_out[i] is None:
            ops_out[i] = ["PASS"]

    return {"farmer": ops_out[0], "hands": ops_out[1:], "market": orders}


def _to_dict(obj):
    """Recursively convert kaggle_environments Struct to plain dict."""
    try:
        import json
        return json.loads(json.dumps(obj))
    except Exception:
        return obj


def agent(obs, config=None):
    try:
        return _agent(_to_dict(obs))
    except Exception as e:
        import traceback
        traceback.print_exc()
        return {"farmer": ["PASS"], "hands": [], "market": []}


# ── Compat exports (used by env_wrapper.py) ───────────────────────────────────
def plan_portfolio(day, total_days, money, free_tiles, seeds):
    """Legacy compat — top strategy doesn't use this."""
    return []


def _count_animals_compat(me, private=None):
    return _count_animals(me, private)


In [ ]:
%%writefile env_wrapper.py
"""Gymnasium wrapper for Kaggriculture — single-agent (vs 'starter')."""
import numpy as np
try:
    import gymnasium as gym
    from gymnasium import spaces
    BaseEnv = gym.Env
except ImportError:
    gym = None
    spaces = None
    BaseEnv = object
from kaggle_environments import make

# ── observation layout (flat float32 vector, len = OBS_DIM) ─────────────────
# day/hour/money/quadrants: 4
# my farm summary (per crop: count, needs_water, harvestable, avg_age): 5*4 = 20
# animal summary (per animal: count, needs_feed, harvestable): 3*3 = 9
# farmer xy: 2
# market prices normalised (9 products): 9
# market inventory normalised (9): 9
# shed contents normalised (9+3 animals): 12
# seeds (5 crops): 5
# town shops unlocked (8 binary): 8
# opponent farm summary (same as mine): 29
# Total: 4+20+9+2+9+9+12+5+8+29 = 107
OBS_DIM = 107

CROPS    = ["WHEAT","CARROT","TOMATO","STRAWBERRY","MELON"]
ANIMALS  = ["GOOSE","COW","SHEEP"]
PRODUCTS = ["WHEAT","CARROT","TOMATO","STRAWBERRY","MELON","EGG","MILK","WOOL","FERTILIZER"]
BASE_PRICES = [25, 35, 60, 120, 250, 50, 160, 200, 100]
SHOPS = ["BAKERY","PIZZA_SHOP","BRUNCH_SPOT","YARN_STORE","ICE_CREAM_SHOP",
         "PET_CAFE","SMOOTHIE_SHOP","FARMERS_MARKET"]

# ── high-level macro-actions for the farmer ──────────────────────────────────
# 0  GO_HARVEST    – move toward / harvest nearest harvestable tile
# 1  GO_WATER      – move toward / water nearest unwatered tile
# 2  GO_PLANT_M    – move toward / plant MELON on nearest empty tile
# 3  GO_PLANT_C    – move toward / plant CARROT
# 4  GO_PLANT_W    – move toward / plant WHEAT
# 5  GO_FEED       – move toward / feed nearest hungry animal
# 6  GO_DIG        – move toward / dig nearest weed
# 7  PASS
N_ACTIONS = 8


def _norm_shop(s):
    return str(s).strip().upper().replace(" ","_").replace("-","_")

def _farm_summary(tiles, day):
    """Returns (20-d crop vec, 9-d animal vec, occupied_count)."""
    cv = np.zeros(20, np.float32)   # 4 features per crop
    av = np.zeros(9,  np.float32)   # 3 features per animal
    occ = 0
    for y, row in enumerate(tiles):
        for x, t in enumerate(row):
            if not isinstance(t, dict):
                continue
            k = t.get("kind")
            if k == "PLANT":
                c = t.get("crop")
                if c in CROPS:
                    ci = CROPS.index(c)
                    cv[ci*4]   += 1                                    # count
                    cv[ci*4+1] += int(not t.get("watered_today",False)) # needs water
                    cv[ci*4+2] += int(t.get("yield_units",0) > 0)      # harvestable
                    cv[ci*4+3] += max(0, day - t.get("planted_day",day)) / 30.0
                occ += 1
            elif k in ("COOP","PASTURE"):
                a = t.get("animal")
                if a in ANIMALS:
                    ai = ANIMALS.index(a)
                    av[ai*3]   += 1
                    av[ai*3+1] += int(not t.get("fed_today",False))
                    av[ai*3+2] += int(t.get("yield_units",0) > 0)
                occ += 1
    return cv, av, occ

def obs_to_vec(obs, player):
    v = np.zeros(OBS_DIM, np.float32)
    me  = obs["farms"][player]
    opp = obs["farms"][1-player]
    day  = obs.get("day",0)
    hour = obs.get("hour",0)
    priv = obs.get("private",{}) or {}
    mkt  = obs.get("market",{}) or {}
    town = obs.get("town",{}) or {}

    i = 0
    v[i] = day / 30.0;   i+=1
    v[i] = hour / 24.0;  i+=1
    v[i] = min(me.get("money",0) / 50000.0, 1.0); i+=1
    v[i] = len(me.get("unlocked_quadrants",["NW"])) / 4.0; i+=1

    tiles = me.get("tiles") or []
    cv, av, _ = _farm_summary(tiles, day)
    v[i:i+20] = cv / 10.0;  i+=20
    v[i:i+9]  = av / 10.0;  i+=9

    fx, fy = me.get("farmer", [4,4])
    v[i] = fx/10.0; v[i+1] = fy/10.0; i+=2

    minv = mkt.get("inventory",{}) or {}
    mprc = mkt.get("prices",{}) or {}
    for j,p in enumerate(PRODUCTS):
        v[i+j] = min(mprc.get(p, BASE_PRICES[j]) / (BASE_PRICES[j]*2), 1.0)
    i+=9
    for j,p in enumerate(PRODUCTS):
        v[i+j] = min(minv.get(p,10000) / 10000.0, 1.0)
    i+=9

    shed = priv.get("shed",{}) or {}
    seeds = priv.get("seeds",{}) or {}
    for j,p in enumerate(PRODUCTS):
        v[i+j] = min(shed.get(p,0) / 20.0, 1.0)
    i+=9
    for j,a in enumerate(ANIMALS):
        v[i+j] = min(shed.get(a,0) / 5.0, 1.0)
    i+=3

    for j,c in enumerate(CROPS):
        v[i+j] = min(seeds.get(c,0) / 10.0, 1.0)
    i+=5

    shops_u = {_norm_shop(s) for s in (town.get("unlocked_shops") or [])}
    for j,s in enumerate(SHOPS):
        v[i+j] = float(s in shops_u)
    i+=8

    opp_tiles = opp.get("tiles") or []
    ocv, oav, _ = _farm_summary(opp_tiles, day)
    v[i:i+20] = ocv / 10.0; i+=20
    v[i:i+9]  = oav / 10.0; i+=9

    assert i == OBS_DIM, f"obs dim mismatch: {i}"
    return v


def _step_toward(pos, target):
    x,y = pos; tx,ty = target
    dx,dy = tx-x, ty-y
    if dx==0 and dy==0: return None
    if abs(dx)>=abs(dy): return "EAST" if dx>0 else "WEST"
    return "SOUTH" if dy>0 else "NORTH"

def _nearest(pos, cells):
    if not cells: return None
    return min(cells, key=lambda c: abs(c[0]-pos[0])+abs(c[1]-pos[1]))

def macro_to_farmer_op(action_id, obs, player, seeds_override=None):
    """Convert macro action id → farmer op list.  Returns (farmer_op, market_list)."""
    me    = obs["farms"][player]
    priv  = obs.get("private",{}) or {}
    seeds = seeds_override or priv.get("seeds",{}) or {}
    tiles = me.get("tiles") or []
    n     = len(tiles)
    pos   = tuple(me.get("farmer",[4,4]))
    day   = obs.get("day",0)

    # scan
    harv, water, empty, weeds, feed = [],[],[],[],[]
    for y in range(n):
        for x in range(n):
            t = tiles[y][x]
            if t=="LOCKED": continue
            if t is None: empty.append((x,y)); continue
            if not isinstance(t,dict): continue
            k = t.get("kind")
            if k=="PLANT":
                if t.get("yield_units",0)>0: harv.append((x,y))
                if not t.get("watered_today",False): water.append((x,y))
            elif k=="WEED": weeds.append((x,y))
            elif k in ("COOP","PASTURE") and t.get("animal"):
                if not t.get("fed_today",False): feed.append((x,y))
                if t.get("yield_units",0)>0: harv.append((x,y))

    def _go(cells, act):
        t = _nearest(pos, cells)
        if t is None: return ["PASS"]
        if tuple(pos)==tuple(t): return [act]
        st = _step_toward(pos,t)
        return [st] if st else [act]

    crop_map = {2:"MELON",3:"CARROT",4:"WHEAT"}
    if action_id == 0: return _go(harv, "HARVEST")
    if action_id == 1: return _go(water, "WATER")
    if action_id in (2,3,4):
        crop = crop_map[action_id]
        if seeds.get(crop,0)>0 and day<=26:
            return _go(empty, f"__PLANT__{crop}")
        return ["PASS"]
    if action_id == 5: return _go(feed, "FEED")
    if action_id == 6: return _go(weeds, "DIG")
    return ["PASS"]  # action 7 = PASS


def resolve_farmer_op(raw_op, obs, player, priv):
    """Turn __PLANT__CROP or regular op into the actual op, handling movement."""
    if not raw_op or raw_op[0]=="PASS": return ["PASS"]
    op = raw_op[0]
    if op.startswith("__PLANT__"):
        crop = op[9:]
        me = obs["farms"][player]
        tiles = me.get("tiles") or []
        pos = tuple(me.get("farmer",[4,4]))
        x,y = pos
        t = tiles[y][x] if 0<=y<len(tiles) and 0<=x<len(tiles[y]) else "LOCKED"
        if t is None:
            return ["PLANT", crop]
        # find nearest empty and move
        empty=[]
        for ry in range(len(tiles)):
            for rx in range(len(tiles[ry])):
                if tiles[ry][rx] is None: empty.append((rx,ry))
        tgt = _nearest(pos, empty)
        if tgt is None: return ["PASS"]
        if tuple(pos)==tgt: return ["PLANT", crop]
        st = _step_toward(pos, tgt)
        return [st] if st else ["PASS"]
    return raw_op


def calc_net_worth(obs, player):
    me = obs["farms"][player]
    priv = obs.get("private", {}) or {}
    money = float(me.get("money", 0.0))
    
    seeds = priv.get("seeds", {}) or {}
    money += seeds.get("MELON", 0) * 80
    money += seeds.get("CARROT", 0) * 20
    money += seeds.get("WHEAT", 0) * 10
    
    shed = priv.get("shed", {}) or {}
    vals = {"MELON": 250, "CARROT": 35, "WHEAT": 25, "TOMATO": 60, "STRAWBERRY": 120}
    for k, v in shed.items():
        money += v * vals.get(k, 50)
        
    tiles = me.get("tiles") or []
    seed_costs = {"MELON": 80, "CARROT": 20, "WHEAT": 10, "TOMATO": 30, "STRAWBERRY": 60}
    for row in tiles:
        for t in row:
            if isinstance(t, dict) and t.get("kind") == "PLANT":
                crop = t.get("crop")
                # Give the full value of the seed, PLUS a bonus for having it in the ground!
                money += seed_costs.get(crop, 20) + (vals.get(crop, 50) * 0.5)
    return money

class KagrEnv(BaseEnv):
    """Single-player Gymnasium env wrapping kaggriculture (player 0 vs 'starter')."""

    metadata = {"render_modes":[]}

    def __init__(self, opponent="starter"):
        if BaseEnv is not object:
            super().__init__()
        self.opponent = opponent
        if spaces is not None:
            self.observation_space = spaces.Box(0.0, 1.0, (OBS_DIM,), np.float32)
            self.action_space = spaces.Discrete(N_ACTIONS)
        self._env = None
        self._obs = None
        self._done = False
        self._prev_money = 3000.0

    def _make_market_orders(self, obs, player=0):
        """Delegate to top-player heuristic strategy."""
        from heuristic import _make_market_orders as h_orders
        return h_orders(obs, player)

    def _buy_seed_if_needed(self, obs, crop, player=0):
        """Return extra market order to buy one seed if we have none."""
        priv = obs.get("private",{}) or {}
        seeds = priv.get("seeds",{}) or {}
        me = obs["farms"][player]
        money = me.get("money",0)
        SEED_COST={"MELON":80,"CARROT":20,"WHEAT":10}
        if seeds.get(crop,0)==0 and money>SEED_COST.get(crop,0)+200:
            return [["BUY_SEED", crop, 3]]
        return []

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._env = make("kaggriculture", debug=False)
        self._trainer = self._env.train([None, self.opponent])
        raw = self._trainer.reset()
        self._obs = dict(raw)
        self._obs.setdefault("player", 0)
        self._prev_money = calc_net_worth(self._obs, 0)
        self._done = False
        return obs_to_vec(self._obs, 0), {}

    def _compose_action(self, action, obs):
        priv  = obs.get("private", {}) or {}
        seeds = priv.get("seeds", {}) or {}
        raw_op    = macro_to_farmer_op(action, obs, 0, seeds)
        farmer_op = resolve_farmer_op(raw_op, obs, 0, priv)
        orders    = self._make_market_orders(obs)
        if raw_op and str(raw_op[0]).startswith("__PLANT__"):
            crop = str(raw_op[0])[9:]
            orders = self._buy_seed_if_needed(obs, crop) + orders
        # hands mirror the farmer's macro so hired help isn't wasted
        n_hands = len(obs["farms"][0].get("hands") or [])
        hands = []
        for _ in range(n_hands):
            hands.append(resolve_farmer_op(raw_op, obs, 0, priv))
        return {"farmer": farmer_op, "hands": hands, "market": orders[:10]}

    def step(self, action):
        if self._done:
            raise RuntimeError("call reset() first")
        obs = self._obs
        act = self._compose_action(int(action), obs)
        new_obs, _kag_reward, done, _info = self._trainer.step(act)
        new_obs = dict(new_obs)
        new_obs.setdefault("player", 0)

        new_money = calc_net_worth(new_obs, 0)
        opp_money = calc_net_worth(new_obs, 1)
        reward = float(new_money - self._prev_money)
        self._prev_money = new_money
        self._obs = new_obs

        if done:
            self._done = True
            reward += 5000.0 if new_obs["farms"][0].get("money", 0) > new_obs["farms"][1].get("money", 0) else -5000.0
        return obs_to_vec(new_obs, 0), reward, bool(done), False, {}


In [ ]:
%%writefile train_rl.py
"""Train PPO agent on Kaggriculture, save weights as numpy for zero-dep inference.

Usage:
    python train_rl.py              # trains for 500k steps (~20 min)
    python train_rl.py --steps 1000000
"""
import argparse, os, sys, numpy as np

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--steps", type=int, default=500_000)
    ap.add_argument("--out",   default="rl_weights.npz")
    ap.add_argument("--n_envs", type=int, default=4)
    args = ap.parse_args()

    from stable_baselines3 import PPO
    from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor
    from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
    from env_wrapper import KagrEnv

    def make_env(rank):
        def _init():
            env = KagrEnv(opponent="starter")
            return env
        return _init

    print(f"Creating {args.n_envs} envs...")
    vec_env = SubprocVecEnv([make_env(i) for i in range(args.n_envs)])
    vec_env = VecMonitor(vec_env)

    eval_env = KagrEnv(opponent="starter")

    model = PPO(
        "MlpPolicy",
        vec_env,
        n_steps=1024,
        batch_size=256,
        n_epochs=10,
        gamma=0.995,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        learning_rate=3e-4,
        policy_kwargs=dict(net_arch=[256, 256, 128]),
        verbose=1,
        tensorboard_log="./rl_logs/",
    )

    checkpoint_cb = CheckpointCallback(
        save_freq=max(50_000 // args.n_envs, 1),
        save_path="./rl_checkpoints/",
        name_prefix="kagr_ppo",
    )
    eval_cb = EvalCallback(
        eval_env,
        n_eval_episodes=3,
        eval_freq=max(50_000 // args.n_envs, 1),
        best_model_save_path="./rl_best/",
        verbose=1,
    )

    print(f"Training for {args.steps} steps...")
    model.learn(args.steps, callback=[checkpoint_cb, eval_cb], progress_bar=False)

    # Save best model as SB3 format
    model.save("rl_final_model")
    print("Saved rl_final_model.zip")

    # Also export weights as numpy for zero-dependency inference
    export_weights(model, args.out)
    print(f"Exported numpy weights → {args.out}")

    vec_env.close()


def export_weights(model, out_path):
    """Extract MLP policy weights into a .npz file for numpy-only inference."""
    policy = model.policy
    params = {}

    # SB3 MlpPolicy stores layers in policy.mlp_extractor and policy.action_net
    state = policy.state_dict()
    for k, v in state.items():
        params[k.replace(".","__")] = v.cpu().numpy()

    np.savez_compressed(out_path, **params)
    print(f"Keys saved: {list(params.keys())[:8]} ...")


if __name__ == "__main__":
    main()


In [ ]:
!python train_rl.py --steps 2000000